# GMRES Preconditioning Study for Frequency Transfers (16→32→64→128)

This notebook benchmarks GMRES for three preconditioning choices:

1. **No preconditioning**
2. **Local CNN transfer preconditioner**
3. **U-Net-style transfer preconditioner**

It includes dataset generation from direct solves, operator training, and all requested plots (iteration counts to `1e-6`, iterate fields `k=0..5`, residual decay). Focus is on the **real part of the current iterate**.


## Design choices (presentation notes)

- Same grid shape across frequencies to avoid transfer shape mismatch.
- PML settings fixed to your accepted setup: `NPML=50`, `eta=4.0`, `power=2`.
- Physics+AI preconditioner:

  
  $M^{-1}(r) \approx L_{\omega/2}^{-1}(\mathcal{T}_\theta(r_{\omega}))$

- `QUICK_MODE` for fast debug; disable for final figures.


In [7]:
# === Diagnose: wat ontbreekt er exact? ===
import sys, importlib, traceback
from pathlib import Path

# ensure src on path
_candidates = [Path.cwd(), *Path.cwd().parents]
_repo = next((p for p in _candidates if (p / "src").exists()), None)
if _repo is None:
    raise RuntimeError(f"Cannot find src/ from cwd={Path.cwd()}")
sys.path.insert(0, str(_repo / "src"))

checks = [
    ("core.config", "HelmholtzConfig"),
    ("core.config", "PMLConfig"),
    ("core.grid", "Grid2D"),
    ("operators.assemble", "assemble_helmholtz_matrix"),
    ("operators.assemble", "ppw_gate"),
    ("core.resolution", "grid_from_ppw_with_pml_extension"),
]

missing = []
for mod_name, sym in checks:
    try:
        m = importlib.import_module(mod_name)
        ok = hasattr(m, sym)
        print(f"{mod_name}.{sym}: {'OK' if ok else 'MISSING'}")
        if not ok:
            missing.append((mod_name, sym))
    except Exception as e:
        print(f"{mod_name}.{sym}: IMPORT ERROR -> {e}")
        missing.append((mod_name, sym))

print("\nMissing/failed:", missing if missing else "none")


core.config.HelmholtzConfig: OK
core.config.PMLConfig: OK
core.grid.Grid2D: OK
operators.assemble.assemble_helmholtz_matrix: OK
operators.assemble.ppw_gate: MISSING
core.resolution.grid_from_ppw_with_pml_extension: OK

Missing/failed: [('operators.assemble', 'ppw_gate')]


In [5]:
# === Bootstrap imports + fallback for missing core.resolution ===
from __future__ import annotations
import sys, types
from pathlib import Path
from dataclasses import dataclass
import numpy as np

# find repo root with src/
_candidates = [Path.cwd(), *Path.cwd().parents]
_repo = next((p for p in _candidates if (p / "src").exists()), None)
if _repo is None:
    raise RuntimeError(f"Could not find src/ from cwd={Path.cwd()}")
sys.path.insert(0, str(_repo / "src"))

# base imports that should always exist
from core.grid import Grid2D

def _install_resolution_fallback():
    """Create in-memory module core.resolution if file is absent in your environment."""
    mod = types.ModuleType("core.resolution")

    def _make_odd(n: int) -> int:
        n = int(n)
        return n if (n % 2 == 1) else (n + 1)

    def grid_from_ppw(
        *,
        omega: float,
        ppw: float,
        lx: float,
        ly: float,
        c_min: float = 1.0,
        n_min: int = 2,
        make_odd: bool = True,
        x_min: float = 0.0,
        y_min: float = 0.0,
    ) -> Grid2D:
        if omega == 0.0:
            raise ValueError("omega must be nonzero.")
        lam_min = 2.0 * np.pi * float(c_min) / abs(float(omega))
        h_target = lam_min / float(ppw)

        nx = max(int(np.ceil(float(lx) / h_target)) + 1, int(n_min))
        ny = max(int(np.ceil(float(ly) / h_target)) + 1, int(n_min))
        if make_odd:
            nx, ny = _make_odd(nx), _make_odd(ny)

        return Grid2D(nx=nx, ny=ny, lx=float(lx), ly=float(ly), x_min=float(x_min), y_min=float(y_min))

    @dataclass(frozen=True)
    class ExtendedGrid2D:
        grid_phys: Grid2D
        grid_ext: Grid2D
        core_slices: tuple[slice, slice]

    def grid_from_ppw_with_pml_extension(
        *,
        omega: float,
        ppw: float,
        lx: float,
        ly: float,
        npml: int,
        c_min: float = 1.0,
        n_min_phys: int = 201,
        make_odd_phys: bool = True,
        x_min_phys: float = 0.0,
        y_min_phys: float = 0.0,
    ) -> ExtendedGrid2D:
        npml = int(npml)
        gphys = grid_from_ppw(
            omega=omega, ppw=ppw, lx=lx, ly=ly, c_min=c_min,
            n_min=n_min_phys, make_odd=make_odd_phys, x_min=x_min_phys, y_min=y_min_phys
        )
        nxp, nyp = int(gphys.nx), int(gphys.ny)
        hx, hy = float(gphys.hx), float(gphys.hy)

        nxe, nye = nxp + 2 * npml, nyp + 2 * npml
        gext = Grid2D(
            nx=nxe, ny=nye,
            lx=float(gphys.lx) + 2.0 * npml * hx,
            ly=float(gphys.ly) + 2.0 * npml * hy,
            x_min=float(x_min_phys) - npml * hx,
            y_min=float(y_min_phys) - npml * hy,
        )
        si = slice(npml, npml + nxp)
        sj = slice(npml, npml + nyp)
        return ExtendedGrid2D(grid_phys=gphys, grid_ext=gext, core_slices=(si, sj))

    mod.grid_from_ppw = grid_from_ppw
    mod.ExtendedGrid2D = ExtendedGrid2D
    mod.grid_from_ppw_with_pml_extension = grid_from_ppw_with_pml_extension
    sys.modules["core.resolution"] = mod
    return mod

# try real module, else inject fallback
try:
    import core.resolution as _res
except Exception:
    _res = _install_resolution_fallback()

from core.resolution import grid_from_ppw_with_pml_extension
print("Resolution provider:", _res.__name__)


Resolution provider: core.resolution


In [6]:
# === Normal project imports ===
from core.config import HelmholtzConfig, PMLConfig
from operators.assemble import assemble_helmholtz_matrix
from operators.solve import solve_on_extended_domain  # this may import core.resolution internally
import scipy.sparse.linalg as spla
import matplotlib.pyplot as plt
import numpy as np

print("Imports OK")


ImportError: cannot import name 'ppw_gate' from 'operators.assemble' (/math/home/fkiewiet/Freq2Transfer/src/operators/assemble.py)

In [ ]:
# === Experiment config ===
QUICK_MODE = True  # Set False for final heavy runs
TRANSFER_PAIRS = [(16.0, 32.0), (32.0, 64.0), (64.0, 128.0)]

LX, LY = 1.0, 1.0
C0 = 1.0
PPW = 10.0

NPML = 50
ETA = 4.0
PML_POWER = 2
N_MIN_PHYS = 129 if QUICK_MODE else 513

N_TRAIN = 8 if QUICK_MODE else 48
N_VAL = 2 if QUICK_MODE else 8
N_BENCH = 3 if QUICK_MODE else 12

N_SOURCES_MIN, N_SOURCES_MAX = 2, 6
AMP_MIN, AMP_MAX = 1.0, 2.0
MARGIN = 0.08

GMRES_TOL = 1e-6
GMRES_MAXITER = 80 if QUICK_MODE else 200

ARTIFACT_DIR = Path('bench_outputs')
ARTIFACT_DIR.mkdir(exist_ok=True, parents=True)


In [ ]:
# === Utilities ===
def make_extended_setup(omega: float):
    ext = grid_from_ppw_with_pml_extension(
        omega=omega, ppw=PPW, lx=LX, ly=LY, npml=NPML,
        c_min=C0, n_min_phys=N_MIN_PHYS, make_odd_phys=True,
        x_min_phys=0.0, y_min_phys=0.0,
    )
    gphys, gext = ext.grid_phys, ext.grid_ext
    si, sj = ext.core_slices

    c_ext = np.full((gext.nx, gext.ny), C0, dtype=float)
    cfg = HelmholtzConfig(
        omega=float(omega), grid=gext,
        pml=PMLConfig(thickness=NPML, strength=ETA, power=PML_POWER),
    )
    A = assemble_helmholtz_matrix(cfg, c_ext).tocsr()
    return gphys, gext, (si, sj), A


def random_sources(rng: np.random.Generator, nsrc: int):
    xs = rng.uniform(MARGIN, 1.0 - MARGIN, size=nsrc)
    ys = rng.uniform(MARGIN, 1.0 - MARGIN, size=nsrc)
    amps = rng.uniform(AMP_MIN, AMP_MAX, size=nsrc)
    phases = rng.uniform(0.0, 2*np.pi, size=nsrc)
    return [dict(x=float(x), y=float(y), amp=float(a), phase=float(p)) for x, y, a, p in zip(xs, ys, amps, phases)]


def rhs_from_sources(gphys, gext, core_slices, sources):
    si, sj = core_slices
    f_phys = np.zeros((gphys.nx, gphys.ny), dtype=np.complex128)
    x = np.asarray(gphys.x)
    y = np.asarray(gphys.y)
    for s in sources:
        ix = int(np.argmin(np.abs(x - s['x'])))
        iy = int(np.argmin(np.abs(y - s['y'])))
        f_phys[ix, iy] += s['amp'] * np.exp(1j * s['phase'])
    f_ext = np.zeros((gext.nx, gext.ny), dtype=np.complex128)
    f_ext[si, sj] = f_phys
    return f_ext.reshape(-1)


def rel_residual(A, x, b):
    r = b - A @ x
    return np.linalg.norm(r) / (np.linalg.norm(b) + 1e-30)


In [ ]:
# Build operators for all frequencies
operator_bank = {}
for w in sorted({w for pair in TRANSFER_PAIRS for w in pair}):
    gphys, gext, core, A = make_extended_setup(w)
    operator_bank[w] = dict(gphys=gphys, gext=gext, core=core, A=A)
    print(f"omega={w:>5.1f} | phys={gphys.nx}x{gphys.ny} | ext={gext.nx}x{gext.ny} | nnz={A.nnz}")


## Dataset generation (direct solves)
For each transfer pair, we randomize 2-6 sources (amplitude 1-2), solve the high-frequency system directly, and build a training target

$y=\Re(A_{\omega/2}u_\omega)$

so the learned transfer predicts a low-frequency RHS proxy.


In [ ]:
def make_pair_dataset(omega_lo, omega_hi, n_samples, seed=0):
    rng = np.random.default_rng(seed)
    Alo = operator_bank[omega_lo]['A']
    Ahi = operator_bank[omega_hi]['A']
    gphys_hi = operator_bank[omega_hi]['gphys']
    gext_hi = operator_bank[omega_hi]['gext']
    core_hi = operator_bank[omega_hi]['core']

    X, Y = [], []
    for _ in range(n_samples):
        ns = int(rng.integers(N_SOURCES_MIN, N_SOURCES_MAX + 1))
        src = random_sources(rng, ns)
        b_hi = rhs_from_sources(gphys_hi, gext_hi, core_hi, src)
        u_hi = spla.spsolve(Ahi, b_hi)
        X.append(np.real(b_hi).astype(np.float32))
        Y.append(np.real(Alo @ u_hi).astype(np.float32))
    return np.stack(X), np.stack(Y)


pair_data = {}
for (wlo, whi) in TRANSFER_PAIRS:
    Xtr, Ytr = make_pair_dataset(wlo, whi, N_TRAIN, seed=int(wlo*10 + 1))
    Xva, Yva = make_pair_dataset(wlo, whi, N_VAL, seed=int(wlo*10 + 2))
    pair_data[(wlo, whi)] = dict(Xtr=Xtr, Ytr=Ytr, Xva=Xva, Yva=Yva)
    print(f"pair {int(wlo)}->{int(whi)} | train={len(Xtr)} val={len(Xva)}")


## Train transfer operators
- Local CNN: one 3x3 kernel (local bias).
- Tiny U-Net: two-level encoder/decoder with skip (global context).


In [ ]:
def conv2d_same(x, k):
    kh, kw = k.shape
    ph, pw = kh // 2, kw // 2
    xp = np.pad(x, ((ph, ph), (pw, pw)), mode='edge')
    out = np.zeros_like(x)
    for i in range(kh):
        for j in range(kw):
            out += k[i, j] * xp[i:i+x.shape[0], j:j+x.shape[1]]
    return out


class LocalCNN:
    def __init__(self):
        self.k = np.zeros((3, 3), dtype=np.float32)
        self.k[1, 1] = 1.0

    def forward(self, x2d):
        return conv2d_same(x2d, self.k)

    def fit(self, X, Y, shape, lr=1e-4, epochs=20):
        H, W = shape
        for ep in range(epochs):
            gk = np.zeros_like(self.k)
            loss = 0.0
            for n in range(len(X)):
                x = X[n].reshape(H, W)
                y = Y[n].reshape(H, W)
                yhat = self.forward(x)
                e = (yhat - y)
                loss += np.mean(e**2)
                xp = np.pad(x, ((1,1),(1,1)), mode='edge')
                for i in range(3):
                    for j in range(3):
                        patch = xp[i:i+H, j:j+W]
                        gk[i, j] += np.mean(e * patch)
            self.k -= lr * (gk / len(X))
            if (ep+1) % 10 == 0:
                print(f"LocalCNN ep {ep+1:03d} | loss={loss/len(X):.4e}")


class TinyUNet:
    def __init__(self, scale=0.05):
        rng = np.random.default_rng(0)
        self.k1 = (rng.standard_normal((3,3))*scale).astype(np.float32)
        self.k2 = (rng.standard_normal((3,3))*scale).astype(np.float32)
        self.k3 = (rng.standard_normal((3,3))*scale).astype(np.float32)

    def _pool2(self, x):
        H, W = x.shape
        H2, W2 = H//2, W//2
        return x[:2*H2, :2*W2].reshape(H2, 2, W2, 2).mean(axis=(1,3))

    def _up2(self, x, out_shape):
        y = np.repeat(np.repeat(x, 2, axis=0), 2, axis=1)
        return y[:out_shape[0], :out_shape[1]]

    def forward(self, x):
        e1 = np.tanh(conv2d_same(x, self.k1))
        p = self._pool2(e1)
        b = np.tanh(conv2d_same(p, self.k2))
        u = self._up2(b, e1.shape)
        cat = u + e1
        return conv2d_same(cat, self.k3)

    def fit(self, X, Y, shape, lr=3e-4, epochs=8, fd_eps=1e-3):
        H, W = shape
        def loss_fn():
            L = 0.0
            for n in range(len(X)):
                x = X[n].reshape(H, W)
                y = Y[n].reshape(H, W)
                L += np.mean((self.forward(x) - y)**2)
            return L / len(X)

        params = [self.k1, self.k2, self.k3]
        for ep in range(epochs):
            base = loss_fn()
            for P in params:
                grad = np.zeros_like(P)
                it = np.nditer(P, flags=['multi_index'], op_flags=['readwrite'])
                while not it.finished:
                    idx = it.multi_index
                    P[idx] += fd_eps
                    lp = loss_fn()
                    P[idx] -= 2*fd_eps
                    lm = loss_fn()
                    P[idx] += fd_eps
                    grad[idx] = (lp - lm) / (2*fd_eps)
                    it.iternext()
                P -= lr * grad
            if (ep+1) % 2 == 0:
                print(f"TinyUNet ep {ep+1:03d} | loss={base:.4e}")


In [ ]:
trained_models = {}
for (wlo, whi), d in pair_data.items():
    shape = (operator_bank[whi]['gext'].nx, operator_bank[whi]['gext'].ny)
    cnn = LocalCNN(); cnn.fit(d['Xtr'], d['Ytr'], shape=shape, epochs=(20 if QUICK_MODE else 60))
    unet = TinyUNet(scale=0.03); unet.fit(d['Xtr'], d['Ytr'], shape=shape, epochs=(8 if QUICK_MODE else 25))
    trained_models[(wlo, whi)] = dict(cnn=cnn, unet=unet)
    print(f"trained {int(wlo)}->{int(whi)}")


In [ ]:
def make_preconditioner(kind, omega_lo, omega_hi, model=None):
    n = operator_bank[omega_hi]['A'].shape[0]
    Alo = operator_bank[omega_lo]['A']
    shape = (operator_bank[omega_hi]['gext'].nx, operator_bank[omega_hi]['gext'].ny)

    if kind == 'none':
        def mv(v):
            return v
    else:
        def mv(v):
            x = np.real(v).reshape(shape)
            y = model.forward(x).reshape(-1)
            z = spla.spsolve(Alo, y.astype(np.complex128))
            return z
    return spla.LinearOperator((n, n), matvec=mv, dtype=np.complex128)


def run_gmres_case(omega_lo, omega_hi, precond_kind, sample_seed=0):
    Ahi = operator_bank[omega_hi]['A']
    gphys = operator_bank[omega_hi]['gphys']
    gext = operator_bank[omega_hi]['gext']
    core = operator_bank[omega_hi]['core']

    rng = np.random.default_rng(sample_seed)
    ns = int(rng.integers(N_SOURCES_MIN, N_SOURCES_MAX + 1))
    src = random_sources(rng, ns)
    b = rhs_from_sources(gphys, gext, core, src)

    model = None
    if precond_kind == 'cnn':
        model = trained_models[(omega_lo, omega_hi)]['cnn']
    elif precond_kind == 'unet':
        model = trained_models[(omega_lo, omega_hi)]['unet']

    M = make_preconditioner('none' if precond_kind == 'none' else 'learned', omega_lo, omega_hi, model)

    residuals, iterates = [], []
    def cb(xk):
        if len(iterates) < 6:
            iterates.append(xk.copy())
        residuals.append(rel_residual(Ahi, xk, b))

    x0 = np.zeros_like(b)
    iterates.append(x0.copy())
    residuals.append(rel_residual(Ahi, x0, b))

    x, info = spla.gmres(
        Ahi, b, x0=x0, M=M,
        rtol=GMRES_TOL, atol=0.0, restart=40, maxiter=GMRES_MAXITER,
        callback=cb, callback_type='x',
    )

    final_rr = rel_residual(Ahi, x, b)
    it_to_tol = next((i for i, rr in enumerate(residuals) if rr <= GMRES_TOL), len(residuals)-1)
    return dict(omega_lo=omega_lo, omega_hi=omega_hi, precond=precond_kind,
                residuals=np.array(residuals), iterates=iterates,
                final_rr=final_rr, info=info, it_to_tol=it_to_tol, nsources=ns)


In [ ]:
# Run benchmark suite
results = []
for (wlo, whi) in TRANSFER_PAIRS:
    for kind in ['none', 'cnn', 'unet']:
        bundle = []
        for k in range(N_BENCH):
            out = run_gmres_case(wlo, whi, kind, sample_seed=1000 + int(whi)*10 + k)
            bundle.append(out)
            results.append(out)
        print(f"{int(wlo)}->{int(whi)} | {kind:>4s} | mean it@1e-6={np.mean([o['it_to_tol'] for o in bundle]):.1f}")


In [ ]:
# Plot: iteration count to convergence
fig, axes = plt.subplots(1, len(TRANSFER_PAIRS), figsize=(15, 4), sharey=True)
for ax, (wlo, whi) in zip(axes, TRANSFER_PAIRS):
    labels = ['none', 'cnn', 'unet']
    vals = []
    for kind in labels:
        xs = [r['it_to_tol'] for r in results if r['omega_lo']==wlo and r['omega_hi']==whi and r['precond']==kind]
        vals.append(np.mean(xs))
    ax.bar(labels, vals)
    ax.set_title(f"{int(wlo)}→{int(whi)}")
    ax.set_ylabel('iterations to 1e-6')
    ax.grid(alpha=0.3)
plt.tight_layout(); plt.savefig(ARTIFACT_DIR / 'iteration_count_to_tol.png', dpi=160); plt.show()


In [ ]:
# Plot: residual decay
fig, axes = plt.subplots(1, len(TRANSFER_PAIRS), figsize=(15, 4), sharey=True)
for ax, (wlo, whi) in zip(axes, TRANSFER_PAIRS):
    for kind, col in [('none','k'), ('cnn','tab:blue'), ('unet','tab:orange')]:
        r = next(rr for rr in results if rr['omega_lo']==wlo and rr['omega_hi']==whi and rr['precond']==kind)
        ax.semilogy(r['residuals'], label=kind, color=col)
    ax.axhline(GMRES_TOL, ls='--', color='red', lw=1)
    ax.set_title(f"Residual decay {int(wlo)}→{int(whi)}")
    ax.set_xlabel('iteration')
    ax.grid(alpha=0.3)
axes[0].set_ylabel('||r_k||/||b||')
axes[-1].legend()
plt.tight_layout(); plt.savefig(ARTIFACT_DIR / 'residual_decay.png', dpi=160); plt.show()


In [ ]:
# Plot: real part fields of iterates 0..5
pair = TRANSFER_PAIRS[-1]  # default 64->128
wlo, whi = pair
shape = (operator_bank[whi]['gext'].nx, operator_bank[whi]['gext'].ny)

fig, axes = plt.subplots(3, 6, figsize=(16, 8))
for row, kind in enumerate(['none', 'cnn', 'unet']):
    r = next(rr for rr in results if rr['omega_lo']==wlo and rr['omega_hi']==whi and rr['precond']==kind)
    for k in range(6):
        ax = axes[row, k]
        if k < len(r['iterates']):
            U = np.real(r['iterates'][k].reshape(shape))
            ax.imshow(U, cmap='RdBu_r')
        ax.set_xticks([]); ax.set_yticks([])
        if row == 0:
            ax.set_title(f'iter {k}')
        if k == 0:
            ax.set_ylabel(kind)
plt.suptitle(f'Real part of current iterate, {int(wlo)}→{int(whi)}')
plt.tight_layout(); plt.savefig(ARTIFACT_DIR / 'iterates_real_0_5.png', dpi=160); plt.show()


## Interpretation checklist for your professors

- Compare bars: does preconditioning reduce iterations to `1e-6`?
- Compare residual curves: does the slope improve (faster decay)?
- Compare iterate snapshots: does phase structure emerge earlier with U-Net than Local CNN?

These plots make your design choices explicit and defensible.


In [ ]:
# Save summary table for slides (CSV + printed rows)
import csv
rows = []
for (wlo, whi) in TRANSFER_PAIRS:
    for kind in ['none','cnn','unet']:
        xs = [r for r in results if r['omega_lo']==wlo and r['omega_hi']==whi and r['precond']==kind]
        rows.append(dict(
            transfer=f"{int(wlo)}->{int(whi)}",
            method=kind,
            mean_it_to_1e6=float(np.mean([q['it_to_tol'] for q in xs])),
            mean_final_rr=float(np.mean([q['final_rr'] for q in xs])),
        ))

out_csv = ARTIFACT_DIR / 'summary_table.csv'
with out_csv.open('w', newline='') as f:
    w = csv.DictWriter(f, fieldnames=['transfer','method','mean_it_to_1e6','mean_final_rr'])
    w.writeheader(); w.writerows(rows)

rows
